In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day03" else here
os.chdir(ROOT)

# 노트북을 어디서 열어도 저장소 루트를 기준으로 돌게 맞춘다.
# backend 를 sys.path 에 넣는 이유는 app.rag 를 설치 없이 import 하기 위해서다.
SANDBOX = ROOT / "sandbox" / "w5" / "day03"        # 이 노트북이 결과물을 쓸 자리
MODELS = ROOT.parent / "models"                    # 저장소 밖. 4.3GB 라 커밋하지 않는다

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"


프로젝트 루트  : C:\workspace\hanwha-agent


In [2]:
if not SAMPLES.is_dir():
    print('재료 폴더가 없습니다 — ')
else:
    all_files = [p for p in SAMPLES.rglob('*') if p.is_file()]
    top_files = [p for p in SAMPLES.iterdir() if p.is_file()]
    sub_files = [p for p in (SAMPLES / '_formats').rglob('*') if p.is_file()]
    today = sorted((p.name for p in SAMPLES.glob('DOC-HR-014_*_v2.0.*')))
    print(f'재료 파일 : {len(all_files)}개')
    print(f'  samples 바로 아래 : {len(top_files)}개 · _formats 안 : {len(sub_files)}개')
    print('오늘 읽을 세 형식 :')
    for name in today:
        print(' ', name)

재료 파일 : 30개
  samples 바로 아래 : 24개 · _formats 안 : 6개
오늘 읽을 세 형식 :
  DOC-HR-014_국내출장_여비_규정_v2.0.docx
  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx
  DOC-HR-014_국내출장_여비_규정_v2.0.pdf


In [3]:
from app.rag.local_parsers import parse_local
TARGET = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
doc = parse_local(TARGET)
print(f'블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count}')

블록 233 · 표 7 · 쪽 1


In [4]:
for i, b in enumerate(doc.blocks[:5]):
    head = b.text[:36] + ('…' if len(b.text) > 36 else '')
    print(f'[{i:2d}] {b.kind} | {b.locator} | {head}')
first_table = next((b for b in doc.blocks if b.kind == '표'))
print()
print('첫 표 블록 : locator=' + first_table.locator)
print(' ', first_table.text[:60])


[ 0] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 일반
[ 1] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 국내출장 여비 규정
[ 2] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | v2.0
[ 3] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 문서번호  DOC-HR-014
[ 4] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 시행일    2025-07-01

첫 표 블록 : locator=표1
  | 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |
|---|---|---|---|--


* 블록 233개 전체를 사용하는 건 비효율적

In [5]:
joined = '\n'.join((b.text for b in doc.blocks))
print(joined[4000:4200])

 기준으로 한다.
② 3급 이상의 직원은 업무 수행상 필요한 경우 특실을 이용할 수 있다.
③ 심야 이동으로 침대차를 이용한 경우에는 그 요금을 지급하고 숙박비를 지급하지 아니한다.
제19조(항공기)
① 국내선 항공기는 철도 대비 이동 시간이 2시간 이상 단축되는 경우에 한하여 이용할 수 있다.
② 국내선 항공기의 좌석 등급은 일반석으로 한다.
③ 항공기를


* 블럭 1개만 사용?

In [6]:
# 숙박비가 포함된 블럭 조회
hits = [(i, b) for i, b in enumerate(doc.blocks) if '숙박비' in b.text]
print(f"'숙박비' 가 들어 있는 블록 : {len(hits)}개")

for i, b in hits[:5]:
    line = b.text.splitlines()[0]
    print(f'  [{i:3d}] {len(b.text):3d}자 | {line[:44]}')
if len(hits) > 5:
    print(f'  ... (나머지 {len(hits) - 5}개는 생략)')

# '광역시 숙박비가 얼마까지 되나요?'
picked = next((b for _, b in hits if b.text.startswith(('①', '②')) and '광역시' in b.text))
print()
print('고른 블록 :', picked.text)
print('  이 블록 안에 조 표시가 있나 :', '제' in picked.text)

short = [b for b in doc.blocks if len(b.text) < 30]
print(f'  30자도 안 되는 블록 : {len(short)}개 / {len(doc.blocks)}개')

'숙박비' 가 들어 있는 블록 : 27개
  [  9] 291자 | | 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |
  [ 29]   7자 | 제2절 숙박비
  [ 30]   9자 | 제14조(숙박비)
  [ 32]  13자 | 제16조(숙박비의 예외)
  [ 68]  39자 | 3. "여비"란 출장에 소요되는 일비·식비·숙박비 및 교통비를 말한다.
  ... (나머지 22개는 생략)

고른 블록 : ② 광역시 지역의 숙박비는 1박당 70,000원을 상한으로 한다.
  이 블록 안에 조 표시가 있나 : False
  30자도 안 되는 블록 : 131개 / 233개


In [7]:
def jaccard(x: str, y: str) -> float:
    sx, sy = (set(x.replace(' ', '')), set(y.replace(' ', '')))
    return len(sx & sy) / len(sx | sy)

PAIRS = [('광역시 숙박비 상한', '잠자리 비용 한도'), ('광역시 숙박비 상한', '광역시 숙박비 상한액')]

for left, right in PAIRS:
    shared = sorted(set(left.replace(' ', '')) & set(right.replace(' ', '')))
    shown = shared if len(shared) <= 3 else f'{len(shared)}개'
    print(f"'{left}' vs '{right}'  겹치는 글자 {shown} · 자카드 {jaccard(left, right):.3f}")

'광역시 숙박비 상한' vs '잠자리 비용 한도'  겹치는 글자 ['비', '한'] · 자카드 0.154
'광역시 숙박비 상한' vs '광역시 숙박비 상한액'  겹치는 글자 8개 · 자카드 0.889


In [8]:
from app.rag.local_parsers import parse_local

DOCX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
doc = parse_local(DOCX)

print('연 문서 :', DOCX.name, '· 블록', len(doc.blocks), '개')
print()

def like(word: str) -> int:
    return sum((1 for b in doc.blocks if word in b.text))

for word in ['숙박비', '잠자리', '잠자리 비용']:
    print(f"LIKE '%{word}%' → {like(word)}건")

연 문서 : DOC-HR-014_국내출장_여비_규정_v2.0.docx · 블록 233 개

LIKE '%숙박비%' → 27건
LIKE '%잠자리%' → 0건
LIKE '%잠자리 비용%' → 0건


In [9]:
import math
POINTS = {'광역시 숙박비 상한': [0.9, 0.2], '잠자리 비용 한도': [0.86, 0.28], '출장 신청 절차': [0.1, 0.95]}

def distance(p: list[float], q: list[float]) -> float:
    # zip 은 짧은 쪽에서 멈추므로 차원이 달라도 예외가 나지 않는다.
    # 값이 그냥 나와 버리면 틀린 순위가 검색 결과가 되므로 먼저 막는다.
    if len(p) != len(q):
        raise ValueError(f'차원이 다르다: {len(p)} vs {len(q)}')
    return math.sqrt(sum(((a - b) ** 2 for a, b in zip(p, q))))
    
BASE = '광역시 숙박비 상한'
print('기준 :', BASE, POINTS[BASE])

for text, point in POINTS.items():
    if text == BASE:
        continue
    gap = distance(POINTS[BASE], point)
    print(f"  {text}  {point} → 거리 {gap:.3f}   {('← 가깝다' if gap < 0.5 else '← 멀다')}")


기준 : 광역시 숙박비 상한 [0.9, 0.2]
  잠자리 비용 한도  [0.86, 0.28] → 거리 0.089   ← 가깝다
  출장 신청 절차  [0.1, 0.95] → 거리 1.097   ← 멀다


In [10]:
v2 = [0.9, 0.2]
v3 = [0.6, 0.8, 0.0]
v1024 = [round(math.sin(i), 4) for i in range(1024)]


def size(v: list[float]) -> float:
    return math.sqrt(sum(x * x for x in v))


for label, v in [("2차원", v2), ("3차원", v3), ("1024차원", v1024)]:
    # 문법 - v[:5] 는 앞에서 다섯 개만 잘라 낸 새 리스트다(슬라이싱)
    print(f"{label:<8} : 길이 {len(v):<4} 앞 5개 {v[:5]}")

print()
print(f"크기(‖v‖) : {size(v2):.4f} · {size(v3):.4f} · {size(v1024):.4f}")

2차원      : 길이 2    앞 5개 [0.9, 0.2]
3차원      : 길이 3    앞 5개 [0.6, 0.8, 0.0]
1024차원   : 길이 1024 앞 5개 [0.0, 0.8415, 0.9093, 0.1411, -0.7568]

크기(‖v‖) : 0.9220 · 1.0000 · 22.6283


In [11]:
# 내적 — 같은 자리끼리 곱해서 모두 더한다
def dot(u: list[float], v: list[float]) -> float:
    # 차원이 다른 벡터를 넣으면 zip 이 앞쪽만 곱하고 조용히 값을 돌려준다.
    # 1024 차원 벡터와 768 차원 벡터를 비교하면 1.0 이어야 할 코사인이 0.8660 으로 나온다.
    # 모델을 바꿀 때 적재분을 같이 바꿔야 하는 이유이기도 하다.
    if len(u) != len(v):
        raise ValueError(f'차원이 다르다: {len(u)} vs {len(v)}')
    return sum((a * b for a, b in zip(u, v)))

# 크기 — 제곱합의 제곱근
def norm(u: list[float]) -> float:
    return math.sqrt(dot(u, u))

# 코사인 유사도 — 내적을 두 크기로 나눈다. 길이를 지우고 방향만 남긴다
def cosine(u: list[float], v: list[float]) -> float:
    return dot(u, v) / (norm(u) * norm(v))

u = [0.6, 0.8, 0]
v = [0, 1, 0]

print('u =', u, '  v =', v)
print(f'  내적   : {dot(u, v):+.4f}')
print(f'  크기   : {norm(u):.4f} × {norm(v):.4f}')
print(f'  코사인 : {cosine(u, v):+.4f}')


u = [0.6, 0.8, 0]   v = [0, 1, 0]
  내적   : +0.8000
  크기   : 1.0000 × 1.0000
  코사인 : +0.8000


In [12]:
w = [-0.6, -0.8, 0]    # u 를 정확히 뒤집은 벡터
perp = [0, 0, 1]       # u 와 직각인 벡터 (perpendicular)

CASES = [
    ("u", u,    "똑같다"),
    ("v", v,    "닮았다"),
    ("⊥", perp, "아무 상관 없다"),
    ("w", w,    "정반대"),
]
for label, other, meaning in CASES:
    print(f"cos(u, {label}) = {cosine(u, other):+.4f}    {meaning}")

cos(u, u) = +1.0000    똑같다
cos(u, v) = +0.8000    닮았다
cos(u, ⊥) = +0.0000    아무 상관 없다
cos(u, w) = -1.0000    정반대


In [13]:
# 질문
q = [1.0, 0.0]
# 뜻이 가깝지만 짧은 문서
# - 벡터가 작다.
d1 = [0.9, 0.1]
# 뜻은 덜 가깝지만 긴 문서
# - 벡터가 크다.
d2 = [3.0, 3.0]

# 내적만으로 비교하면 긴 문서가 우세하여 정확도를 떠나서 토큰이 많이 든다.
print('# 나쁜 예 — 내적만')
bad = {'d1': dot(q, d1), 'd2': dot(q, d2)}
print(f"  d1 {bad['d1']:.3f}  vs  d2 {bad['d2']:.3f}   → 이기는 쪽 = {max(bad, key=bad.get)}")

print('# 좋은 예 — 코사인')
good = {'d1': cosine(q, d1), 'd2': cosine(q, d2)}
print(f"  d1 {good['d1']:.3f}  vs  d2 {good['d2']:.3f}   → 이기는 쪽 = {max(good, key=good.get)}")

# 나쁜 예 — 내적만
  d1 0.900  vs  d2 3.000   → 이기는 쪽 = d2
# 좋은 예 — 코사인
  d1 0.994  vs  d2 0.707   → 이기는 쪽 = d1


In [14]:
def unit(v: list[float]) -> list[float]:
    n = norm(v)
    return [x / n for x in v]

u = [0.6, 0.8, 0]
v = [4.0, 3.0, 0]
uh, vh = (unit(u), unit(v))
print(f'‖û‖ = {norm(uh):.1f}   ‖v̂‖ = {norm(vh):.1f}')

gap = abs(dot(uh, vh) - cosine(u, v))
print(f'dot(û, v̂) = {dot(uh, vh):.6f}   cosine(u, v) = {cosine(u, v):.6f}   차이 {gap:.1f}')
print('차이가 1e-12 보다 작은가 :', gap < 1e-12)

‖û‖ = 1.0   ‖v̂‖ = 1.0
dot(û, v̂) = 0.960000   cosine(u, v) = 0.960000   차이 0.0
차이가 1e-12 보다 작은가 : True


In [15]:
BYTES_PER_NUMBER = 4

def mb(dim: int, n_chunks: int) -> float:
    return dim * n_chunks * BYTES_PER_NUMBER / 1024 / 1024

# 문서 1건이 청크 70개 된다는 가정 하에
CASES = [(1024, 70, ''), (1024, 8000, ''), (4096, 70, ''), (4096, 8000, '     ← 4배')]
for dim, n, note in CASES:
    print(f'{dim}차원 × {n:>4}청크 × 4바이트 = {mb(dim, n):>6.2f} MB{note}')
print()
print(f'4096차원은 1024차원의 {mb(4096, 8000) / mb(1024, 8000):.0f}배다')

1024차원 ×   70청크 × 4바이트 =   0.27 MB
1024차원 × 8000청크 × 4바이트 =  31.25 MB
4096차원 ×   70청크 × 4바이트 =   1.09 MB
4096차원 × 8000청크 × 4바이트 = 125.00 MB     ← 4배

4096차원은 1024차원의 4배다


In [16]:
KURE = MODELS / 'KURE-v1'
BGE = MODELS / 'bge-m3'

MODELS_READY = KURE.is_dir() and BGE.is_dir()
if MODELS_READY:
    print("모델 폴더 찾음")
else:
    print("모델 폴더 못 찾음")

모델 폴더 찾음


In [17]:
def show_folder(title, folder):

    files = sorted((p for p in folder.iterdir() if p.is_file()), key=lambda p: p.stat().st_size, reverse=True)
    total = sum((p.stat().st_size for p in folder.rglob('*') if p.is_file()))
    print(f'{title}  (약 {total / 1000000000.0:.2f} GB)')
    for p in files[:5]:
        print(f'  {p.name:<24} {p.stat().st_size / 1000000.0:>8.1f} MB')
if MODELS_READY:
    show_folder('KURE-v1', KURE)
    show_folder('bge-m3 ', BGE)
else:
    print('모델 폴더가 없어 건너뜁니다. 오늘 진도는 그대로 나갑니다.')

KURE-v1  (약 2.29 GB)
  model.safetensors          2271.1 MB
  tokenizer.json               17.1 MB
  tokenizer_config.json         0.0 MB
  special_tokens_map.json       0.0 MB
  config.json                   0.0 MB
bge-m3   (약 2.29 GB)
  pytorch_model.bin          2271.1 MB
  tokenizer.json               17.1 MB
  sentencepiece.bpe.model       5.1 MB
  special_tokens_map.json       0.0 MB
  config.json                   0.0 MB


HF_HUB_OFFLINE은 transformers나 sentence_transformers를 import하기 전에 설정해야 제대로 적용된다.
- 라이브러리가 import될 때 이 값을 읽는다. 

In [18]:
os.environ["HF_HUB_OFFLINE"] = "1" 

print("HF_HUB_OFFLINE=", os.environ["HF_HUB_OFFLINE"])
# 키가 없을 수도 있는 상황이라면 .get()이 더 안전
# - 키가 없어도 에러 대신 None을 돌려줍니다.
print("HF_HUB_OFFLINE=", os.environ.get("HF_HUB_OFFLINE"))

HF_HUB_OFFLINE= 1
HF_HUB_OFFLINE= 1


In [19]:
PAIRS = [
    ("광역시 숙박비 상한", "잠자리 비용 한도", "같은 뜻"),
    ("출장 전에 신청서를 낸다", "출장은 사전 신청이 원칙이다", "같은 뜻"),
    ("법인카드로 결제한다", "법인카드 사용 지침을 따른다", "가까움"),
    ("일비는 하루 단위로 준다", "숙박비는 1박 단위로 준다", "가까움"),
    ("출장 신청서를 낸다", "재택근무를 신청한다", "다름"),
    ("숙박비 상한액", "정보보안 지침 위반", "남남"),
]

CLOSE = {"같은 뜻", "가까움"}

near = sum(1 for a, b, label in PAIRS if label in CLOSE)
print(f"문장쌍 {len(PAIRS)}개 (가까운 쌍 {near} · 먼 쌍 {len(PAIRS) - near})")

for i, (a, b, label) in enumerate(PAIRS, start=1):
    print(f" {i} [{label}] {a}  ↔  {b}")

문장쌍 6개 (가까운 쌍 4 · 먼 쌍 2)
 1 [같은 뜻] 광역시 숙박비 상한  ↔  잠자리 비용 한도
 2 [같은 뜻] 출장 전에 신청서를 낸다  ↔  출장은 사전 신청이 원칙이다
 3 [가까움] 법인카드로 결제한다  ↔  법인카드 사용 지침을 따른다
 4 [가까움] 일비는 하루 단위로 준다  ↔  숙박비는 1박 단위로 준다
 5 [다름] 출장 신청서를 낸다  ↔  재택근무를 신청한다
 6 [남남] 숙박비 상한액  ↔  정보보안 지침 위반


In [20]:
def jaccard(x, y):
    sx, sy = (set(x.replace(' ', '')), set(y.replace(' ', '')))
    return len(sx & sy) / len(sx | sy)
scores = {True: [], False: []}
for i, (a, b, label) in enumerate(PAIRS, start=1):
    value = jaccard(a, b)
    scores[label in CLOSE].append(value)
    print(f' {i} [{label}] 자카드 {value:.3f}')
print()
print(f'가까운 쌍 평균 : {sum(scores[True]) / len(scores[True]):.3f}')
print(f'먼 쌍 평균     : {sum(scores[False]) / len(scores[False]):.3f}')

 1 [같은 뜻] 자카드 0.154
 2 [같은 뜻] 자카드 0.400
 3 [가까움] 자카드 0.312
 4 [가까움] 자카드 0.538
 5 [다름] 자카드 0.308
 6 [남남] 자카드 0.000

가까운 쌍 평균 : 0.351
먼 쌍 평균     : 0.154


In [21]:
from app.rag.local_parsers import parse_local

docx = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
doc = parse_local(docx)

tables = [b for b in doc.blocks if b.kind == '표']
print(f'표 블록 {len(tables)}개')

for b in tables:
    print(f'  {b.locator}  {len(b.text)}자')
print()
print('표4 앞 6줄 :')
for line in tables[3].text.split('\n')[:6]:
    print(line)

표 블록 7개
  표1  291자
  표2  242자
  표3  210자
  표4  378자
  표5  324자
  표6  302자
  표7  296자

표4 앞 6줄 :
| 교통수단 | 등급 | 지급 기준 | 승인 | 비고 |
|---|---|---|---|---|
| 철도(KTX·SRT) | 일반실 | 실비 | 팀장 | 3급 이상은 특실 가능 |
| 일반철도(무궁화·새마을) | 일반실 | 실비 | 팀장 |  |
| 고속·시외버스 | 우등 이하 | 실비 | 팀장 |  |
| 국내선 항공 | 일반석 | 실비 | 본부장 | 제19조 요건 충족 시 |


In [22]:
# 문서의 모든 텍스트 모으기
joined = '\n'.join((b.text for b in doc.blocks))
# 1200 단위로 잘라서 list로 담기 
naive = [joined[i:i + 1200] for i in range(0, len(joined), 1200)]
# 8,505자 -> 8청크
# - 표까지 같이 자르는 문제 발생
print(f'{len(joined):,}자 → {len(naive)} 청크')

for b in tables:
    start = joined.find(b.text)
    end = start + len(b.text)
    first, last = (start // 1200, (end - 1) // 1200)
    mark = '  두 조각으로 갈린다' if first != last else ''
    print(f'  {b.locator}: 원문 위치 {start:4d}~{end:4d} → 청크 #{first}~#{last}{mark}')

8,505자 → 8 청크
  표1: 원문 위치  115~ 406 → 청크 #0~#0
  표2: 원문 위치 6370~6612 → 청크 #5~#5
  표3: 원문 위치 6701~6911 → 청크 #5~#5
  표4: 원문 위치 7035~7413 → 청크 #5~#6  두 조각으로 갈린다
  표5: 원문 위치 7475~7799 → 청크 #6~#6
  표6: 원문 위치 7833~8135 → 청크 #6~#6
  표7: 원문 위치 8191~8487 → 청크 #6~#7  두 조각으로 갈린다


In [23]:
table4 = tables[3]
start = joined.find(table4.text)
end = start + len(table4.text)
cut = 1200 * ((end - 1) // 1200)
print('── 앞 조각 끝부분 ──')
print(joined[start:cut])
print()
print('── 뒤 조각 앞부분 ──')
print(joined[cut:end])
print()
header = '| 교통수단 |'
print('앞 조각에 머리행이 있나 :', header in joined[start:cut])
print('뒤 조각에 머리행이 있나 :', header in joined[cut:end])


── 앞 조각 끝부분 ──
| 교통수단 | 등급 | 지급 기준 | 승인 | 비고 |
|---|---|---|---|---|
| 철도(KTX·SRT) | 일반실 | 실비 | 팀장 | 3급 이상은 특실 가능 |
| 일반철도(무궁화·새마을) | 일반실 | 실비 | 팀장 |  |
| 고속·시외버스 | 우등 이하 | 실비 | 팀장

── 뒤 조각 앞부분 ──
 |  |
| 국내선 항공 | 일반석 | 실비 | 본부장 | 제19조 요건 충족 시 |
| 선박 | 2등실 이상 | 실비 | 팀장 | 도서 지역 출장 |
| 자가용 | — | 1km당 300원 | 팀장(사전) | 유류비·감가상각 포함 |
| 택시 | — | 실비 | 팀장 | 심야·대중교통 미운행 시에 한함 |
| 렌터카 | 중형 이하 | 실비 | 본부장 | 2인 이상 동행 시 |

앞 조각에 머리행이 있나 : True
뒤 조각에 머리행이 있나 : False


In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=0)
pieces = splitter.split_text(joined)

print(f'{len(joined):,}자 → {len(pieces)} 청크')
for b in tables:
    whole = any((b.text in p for p in pieces))
    print(f"  {b.locator}: {('한 조각에 통째로' if whole else '★ 두 조각으로 갈린다')}")
print('조각 하나의 모양 :', type(pieces[0]).__name__)

C:\workspace\hanwha-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


8,505자 → 8 청크
  표1: 한 조각에 통째로
  표2: 한 조각에 통째로
  표3: 한 조각에 통째로
  표4: ★ 두 조각으로 갈린다
  표5: 한 조각에 통째로
  표6: 한 조각에 통째로
  표7: ★ 두 조각으로 갈린다
조각 하나의 모양 : str


In [25]:
LIMIT = 1200

for b in tables:
    print(f"{b.locator}  {len(b.text)}자  넘나 {len(b.text) > LIMIT}")

print()

longest = max(len(b.text) for b in tables)
over = sum(1 for b in tables if len(b.text) > LIMIT)   # 조건에 맞는 것의 개수를 센다
print(f"가장 긴 표 = {longest}자 · LIMIT 을 넘는 표 = {over}개")

표1  291자  넘나 False
표2  242자  넘나 False
표3  210자  넘나 False
표4  378자  넘나 False
표5  324자  넘나 False
표6  302자  넘나 False
표7  296자  넘나 False

가장 긴 표 = 378자 · LIMIT 을 넘는 표 = 0개


In [26]:
import re

text = '제1조(목적) 이 규정은 여비를 정한다. 제8조(일비) 일비는 30,000원으로 한다.'

print('캡처 그룹 ()      ← 우리가 쓸 형태')
print(re.split('(제\\s*\\d+\\s*조(?:의\\s*\\d+)?\\s*(?:\\([^)]{1,30}\\))?)', text))
print('  parts[0] = 머리 · parts[1],[3],… = 조 제목 · parts[2],[4],… = 본문   (홀짝 규칙)')
print()

print('비캡처 (?:)      ← 조 제목이 통째로 사라진다')
print(re.split('(?:제\\s*\\d+\\s*조(?:의\\s*\\d+)?\\s*(?:\\([^)]{1,30}\\))?)', text))
print()

print('안쪽까지 캡처     ← 그룹이 셋이라 홀짝 규칙이 깨진다')
print(re.split('(제\\s*\\d+\\s*조(의\\s*\\d+)?\\s*(\\([^)]{1,30}\\))?)', text))

캡처 그룹 ()      ← 우리가 쓸 형태
['', '제1조(목적)', ' 이 규정은 여비를 정한다. ', '제8조(일비)', ' 일비는 30,000원으로 한다.']
  parts[0] = 머리 · parts[1],[3],… = 조 제목 · parts[2],[4],… = 본문   (홀짝 규칙)

비캡처 (?:)      ← 조 제목이 통째로 사라진다
['', ' 이 규정은 여비를 정한다. ', ' 일비는 30,000원으로 한다.']

안쪽까지 캡처     ← 그룹이 셋이라 홀짝 규칙이 깨진다
['', '제1조(목적)', None, '(목적)', ' 이 규정은 여비를 정한다. ', '제8조(일비)', None, '(일비)', ' 일비는 30,000원으로 한다.']


In [27]:
import importlib
from app.rag import chunker

importlib.reload(chunker)

body = '\n'.join((b.text for b in doc.blocks if b.kind == '조항'))

parts = chunker._split_articles(body)

print(f'조각 {len(parts)}개')

for title, whole in parts[:5]:
    head = whole[:30] + (' …' if len(whole) > 30 else '')
    print(f'  ({title!r}, {head!r})')

조각 74개
  ('', '일반\n국내출장 여비 규정\nv2.0\n문서번호  DOC-H …')
  ('제1조(목적)', '제1조(목적)')
  ('제2조(정의)', '제2조(정의)')
  ('제3조(적용 범위)', '제3조(적용 범위)')
  ('제4조(다른 규정과의 관계)', '제4조(다른 규정과의 관계)')


In [28]:
importlib.reload(chunker)
long_body = '제7조(일비) ' + '출장자는 출장 기간 중 일비를 지급받는다. ' * 60
pieces = chunker._hard_wrap(long_body)
print(f'합성 예제 {len(long_body):,}자 → {len(pieces)}조각')
for i, piece in enumerate(pieces):
    print(f'  [{i}] {len(piece):4d}자')
print()
real_long = [whole for _, whole in chunker._split_articles(body) if len(whole) > 1200]
print(f'실문서에서 1,200자를 넘는 조항 : {len(real_long)}개')

합성 예제 1,448자 → 2조각
  [0] 1183자
  [1]  263자

실문서에서 1,200자를 넘는 조항 : 0개


In [29]:
importlib.reload(chunker)

from app.rag.chunker import chunk, summarize

# ParsedDoc 하나를 청크 목록으로. 표는 블록 그대로, 본문은 조 단위로 묶인다.
chunks = chunk(doc)
print(summarize(chunks))
print()

# locator 를 보는 것이 요점이다. 장·절이 조 앞에 붙어 있어야
# 검색 결과만 보고도 규정 어디인지 알 수 있다 (RAG 11단계 ⑪ Citation).
for c in chunks[:5]:
    print(f'  {c.kind:<4s} | {c.locator:<34s} | {len(c.text):3d}자')
print()

# 표는 자르지 않으므로 파서가 준 locator 를 그대로 쓴다
print('표 청크 locator :', [c.locator for c in chunks if c.kind == '표'])

sizes = [len(c.text) for c in chunks]
print(f'청크 글자 수    : 최대 {max(sizes)} · 최소 {min(sizes)} · 평균 {sum(sizes) / len(sizes):.1f}')


조항 단위 46 + 표 단위 7 = 53 청크

  조항   | DOC-HR-014_국내출장_여비_규정_v2.0         | 114자
  표    | 표1                                 | 291자
  조항   | DOC-HR-014_국내출장_여비_규정_v2.0         | 653자
  조항   | 제1장 제1조(목적)                        | 125자
  조항   | 제1장 제2조(정의) · 별표2                  | 378자

표 청크 locator : ['표1', '표2', '표3', '표4', '표5', '표6', '표7']
청크 글자 수    : 최대 653 · 최소 17 · 평균 156.4


In [30]:
# 목차 줄을 거르는 기준값을 껐을 때와 비교한다.
# 상수를 바꿔 끼우고 바로 되돌린다 — 뒤 셀이 정상값으로 돌아야 하므로.
before = chunker.TOC_BODY_MIN
chunker.TOC_BODY_MIN = 0
off = chunker.chunk(doc)
chunker.TOC_BODY_MIN = before


def report(label, cs):
    """두 결과를 같은 폭으로 나란히 찍는다."""
    articles = sum((1 for c in cs if c.kind == '조항'))
    tables_n = sum((1 for c in cs if c.kind == '표'))
    avg = sum((len(c.text) for c in cs)) / len(cs)
    return f'{label} : 조항 단위 {articles:3d} + 표 단위 {tables_n} = {len(cs):3d} 청크 · 평균 {avg:5.1f}자'


print(report('정상 (TOC_BODY_MIN = 10)', chunks))
print(report('끔   (TOC_BODY_MIN =  0)', off))
print(f'차이 = {len(off) - len(chunks)}개')
print()

# 껐을 때 늘어난 것이 전부 짧은 조각이다. 제목만 있고 내용이 없는 목차 줄이라
# 검색해도 뜻이 없고, 결과 자리만 차지한다.
print(f'30자 미만 청크 = {sum((1 for c in off if len(c.text) < 30))}개 · 가장 짧은 것들 —')
for c in sorted(off, key=lambda c: len(c.text))[:6]:
    tag = f'{c.locator!r},'
    print(f'  ({tag:<16s}{len(c.text)}자)')


정상 (TOC_BODY_MIN = 10) : 조항 단위  46 + 표 단위 7 =  53 청크 · 평균 156.4자
끔   (TOC_BODY_MIN =  0) : 조항 단위  92 + 표 단위 7 =  99 청크 · 평균  83.3자
차이 = 46개

30자 미만 청크 = 42개 · 가장 짧은 것들 —
  ('제1장 제1조(목적)',  7자)
  ('제1장 제2조(정의)',  7자)
  ('제3장 제1절 제11조(일비)',8자)
  ('제3장 제1절 제12조(식비)',8자)
  ('제3장 제3절 제18조(철도)',8자)
  ('제4장 제25조(파견)', 8자)


In [31]:
import re

# 조 단위 조정 셋이 실제로 들었는지 센다. ①②는 0, ③은 조항 대부분이 나와야 한다.

# ① 괄호 제목을 필수로 바꾼 효과.
#    선택이던 때는 "제32조 에 따라 반환한다" 처럼 문장 안에서 다른 조를
#    가리키는 말까지 조 제목으로 잡혔다(실문서에서 27건).
bare = [c for c in chunks if c.kind == '조항' and re.fullmatch(r'제\d+조', c.locator)]
print(f'① 괄호 제목 없는 「제N조」 locator : {len(bare)}개  (0 이어야 한다)')
for c in bare[:4]:
    print(f'   {c.locator:5s} ← {c.text[:30]!r}')

# ② 장 제목 줄을 본문에서 빼는 효과.
#    locator 로 올라갔으므로 본문에 남으면 같은 정보가 두 번 실린다.
#    '.' 는 줄바꿈에 안 걸리므로 줄 전체가 제목인 줄만 센다.
mixed = [c for c in chunks if re.search(r'(?m)^\s*제\d+장.{0,30}$', c.text)]
print(f'② 본문에 장 제목 줄이 남은 청크   : {len(mixed)}개  (0 이어야 한다)')
for c in mixed[:3]:
    print(f'   {c.locator} … {c.text[-22:]!r}')

# ③ 장·절이 조 앞에 붙었나. 이게 붙어야 검색 결과만 보고 규정 어디인지 안다.
with_ch = [c for c in chunks if re.match(r'제\d+장', c.locator)]
articles = [c for c in chunks if c.kind == '조항']
print(f'③ 「제N장」 으로 시작하는 locator  : {len(with_ch)}개 / 조항 {len(articles)}개')
print('   예 :', [c.locator for c in with_ch[:3]])

# ④ 장 밖에 남는 것들 — 문서 머리말, 부칙, 별표·서식.
#    부칙은 '부칙' 을 장 자리에 넣고, 별표·서식은 장 정보를 버리고 별표 이름만 쓴다.
others = [c for c in articles if c not in with_ch]
print(f'④ 장 밖 조항 청크 : {len(others)}개')
for c in others:
    print(f'   {c.locator}')


① 괄호 제목 없는 「제N조」 locator : 0개  (0 이어야 한다)
② 본문에 장 제목 줄이 남은 청크   : 0개  (0 이어야 한다)
③ 「제N장」 으로 시작하는 locator  : 35개 / 조항 46개
   예 : ['제1장 제1조(목적)', '제1장 제2조(정의) · 별표2', '제1장 제3조(적용 범위)']
④ 장 밖 조항 청크 : 11개
   DOC-HR-014_국내출장_여비_규정_v2.0
   DOC-HR-014_국내출장_여비_규정_v2.0
   부칙 제1조(시행일)
   부칙 제2조(경과조치)
   부칙 제3조(다른 규정의 폐지)
   별표1
   별표2
   별표3
   별지 제1호 서식
   별지 제1호 서식
   별지 제2호 서식


In [32]:
importlib.reload(chunker)
from app.rag.chunker import chunk, summarize   

chunks = chunk(doc)            
print(summarize(chunks))

bare = [c for c in chunks if c.kind == "조항" and re.fullmatch(r"제\d+조", c.locator)]
mixed = [c for c in chunks if re.search(r"\n\s*제\d+장", c.text)]
print(f"괄호 제목 없는 「제N조」 locator : {len(bare)}개")
print(f"본문에 장 제목 줄이 섞인 청크 : {len(mixed)}개")
print(f"「제N장」 으로 시작하는 locator : {sum(bool(re.match(r'제\d+장', c.locator)) for c in chunks)}개")
print()

for c in chunks:
    if c.locator.startswith(("제3장 제2절 제14조", "제4장 제26조", "부칙 제1조")):
        print(f"   {c.locator:24s} … {c.text[-22:]!r}")


조항 단위 46 + 표 단위 7 = 53 청크
괄호 제목 없는 「제N조」 locator : 0개
본문에 장 제목 줄이 섞인 청크 : 0개
「제N장」 으로 시작하는 locator : 35개

   제3장 제2절 제14조(숙박비) · 별표1  … '영수증을 첨부한 경우에 한하여 지급한다.'
   제4장 제26조(재해 대응 출장)       … '경우 복귀 후 사유서를 제출하여야 한다.'
   부칙 제1조(시행일)              … '정은 2025년 7월 1일부터 시행한다.'


In [33]:
# 테스트 파일을 노트북에서 들여다본다. 합성 ParsedDoc 으로 거의 다 돌기 때문에
# 실습 재료가 없어도 세 개는 그대로 통과한다.
path = ROOT / "backend" / "tests" / "test_chunker.py"
text = path.read_text(encoding="utf-8")
print(path.name, "·", len(text.splitlines()), "줄")
print("  테스트 함수 :", text.count("\ndef test_"))

test_chunker.py · 68 줄
  테스트 함수 : 4


In [34]:
# 실제로 돌려 본다. 노트북에서 ! 를 붙이면 셸 명령이 된다.
# python -m pytest 로 부르는 이유는 가상환경의 pytest 를 확실히 쓰기 위해서다.
!python -m pytest backend/tests/test_chunker.py -q

....                                                                     [100%]
============================== warnings summary ===============================
.venv\Lib\site-packages\starlette\testclient.py:53
  C:\workspace\hanwha-agent\.venv\Lib\site-packages\starlette\testclient.py:53: DeprecationWarning: The anyio.abc.BlockingPortal alias is deprecated, use anyio.from_thread.BlockingPortal instead.
    _PortalFactoryType = Callable[[], AbstractContextManager[anyio.abc.BlockingPortal]]

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
4 passed, 1 warning in 7.95s


In [35]:
# 같은 규정을 세 형식으로 읽어 청킹까지 돌린다.
# 파서가 다르면 청크도 달라지는지 보는 것이 목적이다.
from app.rag.local_parsers import parse_local
from app.rag.chunker import chunk, summarize

parsed, chunks = ({}, {})
for ext in ('hwpx', 'docx', 'pdf'):
    path = next(SAMPLES.glob(f'DOC-HR-014_*_v2.0.{ext}'))
    doc = parse_local(path)
    cs = chunk(doc)
    parsed[ext], chunks[ext] = (doc, cs)
    print(f'[{ext:4}] blocks={len(doc.blocks):3} tables={doc.table_count} pages={doc.page_count:2} → {summarize(cs)}')

[hwpx] blocks=233 tables=7 pages= 1 → 조항 단위 46 + 표 단위 7 = 53 청크
[docx] blocks=233 tables=7 pages= 1 → 조항 단위 46 + 표 단위 7 = 53 청크
[pdf ] blocks= 16 tables=0 pages=16 → 조항 단위 45 + 표 단위 0 = 45 청크


In [36]:
# hwpx 와 docx 는 같은데 pdf 만 다르다. 왜 다른지는 블록 종류를 보면 안다.
for ext in ('docx', 'pdf'):
    doc = parsed[ext]
    kinds = sorted({b.kind for b in doc.blocks})
    # 표를 마크다운으로 바꿀 때 쓴 | 기호를 센다. 0 이면 표로 인식된 것이 없다는 뜻이다.
    bars = '\n'.join((b.text for b in doc.blocks)).count('|')
    print(f'{ext:4} 블록 종류 : {str(kinds):16} 마크다운 표 기호 | : {bars}개')
print()

# pdf 에서 표 내용이 어디로 갔나. 사라진 것이 아니라 조항 청크 안의 평문이 됐다.
for ext in ('docx', 'pdf'):
    hit = [c for c in chunks[ext] if '70,000' in c.text]
    print(f'{ext:4} 숙박비 상한액이 들어간 청크 {len(hit)}개')
    for c in hit:
        print(f'     {c.kind:<4s} | {c.locator}')

# 표가 어떤 모양으로 남는지 한 청크를 펴 본다.
# 행과 열이 줄바꿈으로만 구분돼서 '직급 / 일비 / 숙박비(광역시)' 가 한 줄로 흐른다.
target = next((c for c in chunks['pdf'] if '숙박비(광역시)' in c.text))
print()
print(f'pdf 쪽 청크 : kind={target.kind} · locator={target.locator} · {len(target.text)}자')
print('  ' + target.text[:80].replace('\n', ' ⏎ ') + ' …')


docx 블록 종류 : ['조항', '표']      마크다운 표 기호 | : 323개
pdf  블록 종류 : ['조항']           마크다운 표 기호 | : 0개

docx 숙박비 상한액이 들어간 청크 3개
     조항   | 제3장 제2절 제14조(숙박비) · 별표1
     표    | 표2
     표    | 표3
pdf  숙박비 상한액이 들어간 청크 3개
     조항   | 제3장 제2절 제14조(숙박비) · 별표1
     조항   | 별표1
     조항   | 별표2

pdf 쪽 청크 : kind=조항 · locator=별표1 · 281자
  [별표1] 직급별 여비 지급 기준표 (제5조제2항, 제11조제3항, 제12조제2항, 제14조제1항 관련) ⏎ 직급 ⏎ 일비 ⏎ 숙박비(광역시) ⏎ 숙박비(그  …
